# Aufteilung der Medienbeiträge

Zwei Datensätze, **dieselbe Auswertung**:

1. **Rohdaten:** `FSI_Heilbronn_Medienbeiträge (1).json` (übergeordneter Ordner)
2. **Clean-Daten:** `clean_data.json` (gefiltert mit `1_filter_data.py`)

Pro Datensatz: Sprache, Kanäle, Social-Plattformen, YouTube-Split. Zum Schluss Kurzvergleich Roh vs. Clean.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

DATEN_NAME = "FSI_Heilbronn_Medienbeiträge (1).json"
CLEAN_NAME = "clean_data.json"

SOCIAL_RELEVANT = frozenset({
    "YouTube", "Instagram", "Facebook", "Mastodon", "Flickr", "Reddit", "LinkedIn", "TikTok", "X",
})

PLATTFORM_DOMAINS = (
    ("YouTube", ("youtube.com", "youtu.be")),
    ("Instagram", ("instagram.com",)),
    ("Facebook", ("facebook.com",)),
    ("Reddit", ("reddit.com",)),
    ("LinkedIn", ("linkedin.com",)),
    ("TikTok", ("tiktok.com",)),
    ("Telegram", ("t.me", "telegram.me", "telegram.org")),
    ("Flickr", ("flickr.com",)),
    ("Pinterest", ("pinterest.com",)),
    ("Tumblr", ("tumblr.com",)),
    ("Bluesky", ("bsky.app",)),
    ("Vimeo", ("vimeo.com",)),
    ("X", ("twitter.com", "x.com")),
)

URLAUB_HOST_MARKEN = (
    "erlebnisrundreisen", "holidaycheck", "urlaubspiraten", "hometravel", "phototravel",
    "alpinforum", "skipass", "ski-nordique", "bergrettung", "22places", "kommwirmachendas",
    "urlaub", "partir.", "tui.com", "mein-mmo", "flugreisende",
)

REGIONAL_HOST_MARKEN = (
    "meinbezirk", "bezirk", "anzeiger", "rundblick", "24hamburg", "bw24", "soester-anzeiger",
    "sauerlandkurier", "salzkammergut", "rheingau", "az-online", "wmn.de", "it-boltwise",
    "vol.at", "krone.at", "kurier.at",
)

KANAL_REIHENFOLGE = (
    "Zeitungen und klassische Medien",
    "Social Media",
    "Urlaub",
    "Regionalnachrichten",
    "Andere Webquellen",
)
SPRACHEN = ("Deutsch", "Französisch")
FARBEN = {"Deutsch": "#e07a3d", "Französisch": "#3d7ec9"}


def finde_datei(dateiname):
    hier = Path.cwd().resolve()
    for ordner in (hier, hier.parent, hier.parent.parent):
        kandidat = ordner / dateiname
        if kandidat.is_file():
            return kandidat
    raise FileNotFoundError(f"{dateiname} nicht gefunden (Start im Ordner Hackathon-KJRJ).")


ROH_PFAD = finde_datei(DATEN_NAME)
CLEAN_PFAD = finde_datei(CLEAN_NAME)
print("Rohdaten:", ROH_PFAD)
print("Clean-Daten:", CLEAN_PFAD)


def host_aus_quelle(quelle):
    host = str(quelle or "").strip().lower()
    if host.startswith("www."):
        host = host[4:]
    return host


def host_ist(host, domain):
    return host == domain or host.endswith("." + domain)


def plattform_roh(quelle, medientyp):
    host = host_aus_quelle(quelle)
    for name, domains in PLATTFORM_DOMAINS:
        if any(host_ist(host, domain) for domain in domains):
            return name
    if medientyp == "Microblog":
        return "Mastodon"
    if medientyp in {"YouTube Video", "YouTube Comments"}:
        return "YouTube"
    return None


def plattform_bestimmen(quelle, medientyp):
    name = plattform_roh(quelle, medientyp)
    if name in SOCIAL_RELEVANT:
        return name
    return None


def sprache_bestimmen(code):
    code = str(code or "").lower()
    if code.startswith("de"):
        return "Deutsch"
    if code.startswith("fr"):
        return "Französisch"
    return "Andere"


def urlaub_bestimmen(host, medientyp, titel, inhalt):
    text = f"{titel}\n{inhalt}".lower()[:2500]
    if any(marke in host for marke in URLAUB_HOST_MARKEN):
        return True
    if medientyp == "Forums":
        return True
    if medientyp == "Pictures" and "infoclimat" not in host:
        if any(w in text for w in ("wander", "ski", "urlaub", "reise", "bergbahn", "gondel")):
            return True
    treffer = sum(1 for w in ("urlaub", "reise", "wanderung", "skigebiet", "ferien", "bergfahrt", "hotel") if w in text)
    if treffer >= 2:
        return True
    return "urlaub" in text and medientyp in {"Blog", "Other"}


def regional_bestimmen(host, medientyp):
    if medientyp == "News":
        return False
    if any(marke in host for marke in REGIONAL_HOST_MARKEN):
        return True
    return medientyp in {"Other", "Comments", "Blog"} and any(
        marke in host for marke in ("krone.at", "vol.at", "kurier.at", "meinbezirk")
    )


def kanal_bestimmen(medientyp, plattform, host, titel, inhalt):
    if isinstance(plattform, str) and plattform:
        return "Social Media"
    if medientyp == "News":
        return "Zeitungen und klassische Medien"
    if urlaub_bestimmen(host, medientyp, titel or "", inhalt or ""):
        return "Urlaub"
    if regional_bestimmen(host, medientyp):
        return "Regionalnachrichten"
    return "Andere Webquellen"


def mit_summe(tabelle):
    ergebnis = tabelle.copy()
    ergebnis["Summe"] = ergebnis.sum(axis=1)
    ergebnis.loc["Summe"] = ergebnis.sum(axis=0)
    return ergebnis.astype(int)


def als_text(tabelle):
    def zelle(wert):
        return f"{int(wert):,}".replace(",", ".")

    if isinstance(tabelle, pd.Series):
        return tabelle.map(zelle)
    return tabelle.apply(lambda spalte: spalte.map(zelle))


def achsen_tausender(achse, _position):
    return f"{int(achse):,}".replace(",", ".")


def dataframe_aufbereiten(pfad):
    roh = pd.read_json(pfad, orient="records")
    zeilen = []
    for eintrag in roh.itertuples(index=False):
        host = host_aus_quelle(eintrag.source_name)
        plattform = plattform_bestimmen(eintrag.source_name, eintrag.media_type_name)
        zeilen.append(
            {
                "sprache": sprache_bestimmen(eintrag.language_code),
                "plattform": plattform,
                "plattform_hinweis": plattform_roh(eintrag.source_name, eintrag.media_type_name),
                "kanal": kanal_bestimmen(
                    eintrag.media_type_name,
                    plattform,
                    host,
                    eintrag.title,
                    eintrag.content,
                ),
            }
        )
    meta = pd.DataFrame(zeilen, index=roh.index)
    spalten = [s for s in roh.columns if s not in meta.columns]
    return pd.concat([meta, roh[spalten]], axis=1)


def ausschnitt(df, sprache, kanal=None, plattform=None):
    auswahl = df[df["sprache"].eq(sprache)]
    if kanal is not None:
        auswahl = auswahl[auswahl["kanal"].eq(kanal)]
    if plattform is not None:
        auswahl = auswahl[auswahl["plattform"].eq(plattform)]
    return auswahl


def auswertung(df, titel):
    sep = "=" * 72
    print(f"\n{sep}\n{titel} — {len(df)} Beiträge\n{sep}")

    sprache_anzahl = df["sprache"].value_counts().rename("Beiträge")
    sprache_anzahl = sprache_anzahl.reindex(["Deutsch", "Französisch", "Andere"], fill_value=0)
    print("\n--- Sprache ---")
    print(als_text(sprache_anzahl.to_frame()).to_string())

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    balken = sprache_anzahl.loc[list(SPRACHEN)]
    balken.plot(kind="bar", ax=ax, color=[FARBEN[n] for n in balken.index], width=0.7)
    ax.set_title(f"{titel}: Beiträge nach Sprache")
    ax.set_ylabel("Beiträge")
    ax.yaxis.set_major_formatter(FuncFormatter(achsen_tausender))
    fig.tight_layout()
    plt.show()

    teil = df[df["sprache"].isin(SPRACHEN)]
    kanal_anzahl = (
        pd.crosstab(teil["kanal"], teil["sprache"])
        .reindex(index=list(KANAL_REIHENFOLGE), columns=list(SPRACHEN), fill_value=0)
    )
    print("\n--- Kanäle ---")
    print(als_text(mit_summe(kanal_anzahl)).to_string())

    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    kanal_anzahl.plot(kind="bar", ax=ax, color=[FARBEN[n] for n in kanal_anzahl.columns], width=0.75)
    ax.set_title(f"{titel}: Kanal nach Sprache")
    ax.set_ylabel("Beiträge")
    ax.tick_params(axis="x", rotation=18)
    ax.yaxis.set_major_formatter(FuncFormatter(achsen_tausender))
    fig.tight_layout()
    plt.show()

    for name in ("Urlaub", "Regionalnachrichten", "Andere Webquellen"):
        teilmenge = df[df["sprache"].isin(SPRACHEN) & df["kanal"].eq(name)]
        tabelle = (
            pd.crosstab(teilmenge["media_type_name"], teilmenge["sprache"])
            .reindex(columns=list(SPRACHEN), fill_value=0)
            .sort_values("Deutsch", ascending=False)
        )
        print(f"\n--- {name} (Medientyp) ---")
        if teilmenge.empty:
            print("(keine Beiträge)")
        else:
            print(als_text(mit_summe(tabelle)).to_string())

    sozial = df[df["kanal"].eq("Social Media") & df["sprache"].isin(SPRACHEN)]
    plattform_anzahl = pd.crosstab(sozial["plattform"], sozial["sprache"]).reindex(columns=list(SPRACHEN), fill_value=0)
    plattform_anzahl["Summe"] = plattform_anzahl.sum(axis=1)
    plattform_anzahl = plattform_anzahl.sort_values("Summe", ascending=False)
    print("\n--- Social Media (relevante Plattformen) ---")
    if sozial.empty:
        print("(keine Beiträge)")
    else:
        print(als_text(mit_summe(plattform_anzahl.astype(int))).to_string())
        plot_plattform = plattform_anzahl.drop(columns="Summe").sort_values("Deutsch", ascending=True)
        fig, ax = plt.subplots(figsize=(8.4, 5.0))
        plot_plattform.plot(kind="barh", ax=ax, color=[FARBEN[n] for n in plot_plattform.columns], width=0.75)
        ax.set_title(f"{titel}: Social-Media-Quellen")
        ax.set_xlabel("Beiträge")
        ax.xaxis.set_major_formatter(FuncFormatter(achsen_tausender))
        fig.tight_layout()
        plt.show()

    ausgeschlossen = (
        df[df["plattform"].isna() & df["plattform_hinweis"].notna()]
        .groupby("plattform_hinweis")
        .size()
        .sort_values(ascending=False)
        .rename("Beiträge")
    )
    print("\n--- Ausgeschlossene Social-Plattformen ---")
    if ausgeschlossen.empty:
        print("(keine)")
    else:
        print(ausgeschlossen.to_string())

    youtube = df[df["plattform"].eq("YouTube") & df["sprache"].isin(SPRACHEN)].copy()
    if not youtube.empty:
        youtube["beitragsart"] = youtube["media_type_name"].map(
            {"YouTube Video": "Video", "YouTube Comments": "Kommentar"}
        ).fillna("Sonstige")
        youtube_anzahl = pd.crosstab(youtube["beitragsart"], youtube["sprache"]).reindex(
            index=["Video", "Kommentar", "Sonstige"], columns=list(SPRACHEN), fill_value=0
        )
        youtube_anzahl = youtube_anzahl.loc[youtube_anzahl.sum(axis=1).gt(0)]
        print("\n--- YouTube ---")
        print(als_text(mit_summe(youtube_anzahl)).to_string())

    irrelevant = df["plattform"].isna() & df["plattform_hinweis"].notna()
    print("\nSocial ohne Relevanz-Schwelle:", int(irrelevant.sum()))

    if "matched_projects" in df.columns:
        projekte = df["matched_projects"].dropna().explode().value_counts().rename("Beiträge")
        if not projekte.empty:
            print("\n--- Projekte (Clean-Daten) ---")
            print(projekte.to_string())



## Rohdaten


In [ ]:
df_roh = dataframe_aufbereiten(ROH_PFAD)
df = df_roh
auswertung(df_roh, "Rohdaten")
df_roh[["sprache", "kanal", "plattform", "title"]].head(8)



## Clean-Daten

Gefiltert: projektbezogene Treffer, ohne Tourismus-Beifang. Spalte `matched_projects` zeigt das Projekt.


In [ ]:
df_clean = dataframe_aufbereiten(CLEAN_PFAD)
df = df_clean
auswertung(df_clean, "Clean-Daten")
df_clean[["sprache", "kanal", "plattform", "matched_projects", "title"]].head(8)



## Vergleich Roh vs. Clean


In [ ]:
vergleich = pd.DataFrame(
    {
        "Rohdaten": df_roh["kanal"].value_counts(),
        "Clean-Daten": df_clean["kanal"].value_counts(),
    }
).fillna(0).astype(int)
vergleich = vergleich.reindex(index=list(KANAL_REIHENFOLGE), fill_value=0)
vergleich["Anteil Clean an Roh (%)"] = (
    vergleich["Clean-Daten"] / vergleich["Rohdaten"].replace(0, pd.NA) * 100
).round(1)
print(als_text(vergleich).to_string())

